# Privacy and de-identification: HIPAA, GDPR and India's DPDP Act

Exercise notebook for the lesson **[Privacy and de-identification: HIPAA, GDPR and India's DPDP Act](https://ramkumarmd.com/medical-ai/foundations/privacy-and-de-identification)** in the *Foundations* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Removing names is not enough to make health data anonymous: combinations of ordinary details such as age, sex and postcode can identify most people. Privacy laws treat this differently. HIPAA sets two routes to de-identification, the GDPR still treats pseudonymised data as personal data for anyone who can re-identify it, and India's DPDP Act applies to digital personal data with a research exemption that operates from May 2027. Good practice combines technical methods with legal agreements and limited access.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Measuring risk with k-anonymity

Check k-anonymity on age, sex and postcode in a synthetic dataset.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 1000
df = pd.DataFrame({
    "age": rng.integers(18, 90, size=n),
    "sex": rng.choice(["F", "M"], size=n),
    "postcode": rng.choice([f"5000{i:02d}" for i in range(40)], size=n),
    "diagnosis": rng.choice(["T2DM", "Hypertension", "CKD", "None"], size=n),
})
quasi_identifiers = ["age", "sex", "postcode"]

def k_anonymity(data, cols):
    sizes = data.groupby(cols).size()
    return sizes.min(), (sizes == 1).sum()

k, unique = k_anonymity(df, quasi_identifiers)
print(f"Raw data:     k = {k}, people unique on age+sex+postcode = {unique}")

# Raw data:     k = 1, people unique on age+sex+postcode = 840

### Reducing risk: generalisation and date shifting

Continuing from the previous block: generalise age and postcode, then re-check.

In [ ]:
generalised = df.assign(
    age=(df.age // 10 * 10).astype(str) + "s",   # 47 -> "40s"
    postcode=df.postcode.str[:4] + "**",          # 500017 -> "5000**"
)
k, unique = k_anonymity(generalised, quasi_identifiers)
print(f"Generalised:  k = {k}, people unique on age+sex+postcode = {unique}")
print(generalised.head(3))

# Generalised:  k = 14, people unique on age+sex+postcode = 0
#    age sex postcode     diagnosis
# 0  20s   F   5000**  Hypertension
# 1  20s   F   5000**           CKD
# 2  70s   F   5000**           CKD

### Shifting dates without losing time intervals

One random shift per patient, applied to every one of their dates.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
visits = pd.DataFrame({
    "patient_id": ["p1", "p1", "p2", "p2"],
    "visit_date": pd.to_datetime(["2026-01-10", "2026-03-02", "2026-02-15", "2026-02-20"]),
})
shifts = {pid: pd.Timedelta(days=int(rng.integers(-180, 181)))
          for pid in visits.patient_id.unique()}
visits["shifted_date"] = visits.visit_date + visits.patient_id.map(shifts)
visits["days_since_first"] = (visits.groupby("patient_id").shifted_date
                              .transform(lambda d: (d - d.min()).dt.days))
print(visits)

#   patient_id visit_date shifted_date  days_since_first
# 0         p1 2026-01-10   2026-05-02                 0
# 1         p1 2026-03-02   2026-06-22                51
# 2         p2 2026-02-15   2025-09-18                 0
# 3         p2 2026-02-20   2025-09-23                 5

## Exercise

**Goal:** Measure and reduce re-identification risk in a synthetic patient table, and see what each step costs in analytical detail.

**Data:** Synthetic patients generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Generate the 1,000-patient table and compute k-anonymity for age + sex + postcode.

In [ ]:
# Your code here


### Step 2

Try three generalisation schemes (5-year bands, 10-year bands, 3-digit postcodes) and record k and the number of unique people for each.

In [ ]:
# Your code here


### Step 3

For your preferred scheme, check l-diversity: does any group have only one diagnosis?

In [ ]:
# Your code here


### Step 4

Apply per-patient date shifting to a synthetic visit table and confirm intervals are preserved.

In [ ]:
# Your code here


### Step 5

Write a short note: which safeguards beyond de-identification would you add before sharing this data with an external AI developer?

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/foundations/privacy-and-de-identification](https://ramkumarmd.com/medical-ai/foundations/privacy-and-de-identification). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.